# Clustering and Annotation

We performed data normalization, dimensionality reduction, and batch correction  
to eliminate technical variability among samples, followed by clustering and cell-type annotation  
to identify distinct cellular populations.  
All processes were conducted using the **OmicVerse** framework in *Python*.

In [ ]:
import scanpy as sc
import omicverse as ov
import numpy as np
import celltypist
from celltypist import models
import seaborn as sns
import matplotlib.pyplot as plt
ov.plot_set(dpi=150,font_path='Arial')

# Enable auto-reload for development
%load_ext autoreload
%autoreload 2

In [ ]:
ov.settings.cpu_init()

In [ ]:
# AnnData Loading
adata_raw = sc.read_h5ad(r"F:\Code\codex\PlaceCell_snRNAseq\Data\FilteredData\All_QC_filtered_annotation_20260604.h5ad")
adata_raw

In [ ]:
adata = adata_raw.copy()
adata

In [ ]:
sc.pl.violin(
    adata,
    ["detected_genes", "nUMIs", "mito_perc"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
fig = ov.pl.embedding(
    adata,
    basis='X_mde',
    color=['sample'],
    frameon='small',
    show=False,
    return_fig=True
)

fig.savefig("embedding_sample.svg", bbox_inches="tight")
fig.savefig("embedding_sample.png", dpi=300, bbox_inches="tight")

In [ ]:
marker_genes = {

    "Neuron": ['Rbfox3','Meg3','Syt1','Snap25'],
    # 兴奋性神经元（Cortical excitatory neurons）
    "Excitatory Neuron":     ["Slc17a7", "Sv2b", "Neurod2"],

    # 抑制性神经元（GABAergic interneurons）
    "GABAergic Neuron":      ["Gad1", "Gad2", "Slc32a1"],

    # 星形胶质细胞（Astrocytes）
    "Astrocyte":                   ["Gja1", "Aldoc", "Mfge8","Cldn10","Aqp4","Acsbg1","Gfap"],

    # # 少突胶质细胞及前体（Oligodendrocyte lineage）
    "OPC":                         ["Pdgfra", "Cspg4", "C1ql1","Tnfaip2"],
    "Oligodendrocyte":             ["Cldn11",     "Plp1",    "Mog"],

    # # 小胶质细胞（Microglia）
    "Microglia":                   ["Cx3cr1",  "C1qa",  "C1qb", "C1qc", "Tmem119", "Ctss"],

    # # 血管相关细胞（Vascular cells）
    "Endothelial":                 ["Rgs5",  "Cldn5", "Ly6a"],

    # # 其它（可按需添加）
    "Ependymal":                   ["Folr1",   "Kcnj13",  "Sostdc1"],
}


In [ ]:
from matplotlib.colors import LinearSegmentedColormap
colors = ["#22456F", "#528FAC","#AECAD8","#F2F2F2","#F7D29D","#EEB84B", "#E87A2D"]  # 示例颜色
cmap_custom = LinearSegmentedColormap.from_list("custom_cmap", colors, N=256)

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# SVG 文字在 AI 中可编辑
mpl.rcParams["svg.fonttype"] = "none"

# =========================
# 1. sample 离散配色
# =========================
sample_key = "sample"

# 推荐：Tableau 10 风格，适合 10 个离散 sample
sample_colors = [
    "#4E79A7",  # blue
    "#F28E2B",  # orange
    "#59A14F",  # green
    "#E15759",  # red
    "#B07AA1",  # purple
    "#9C755F",  # brown
    "#EDC948",  # yellow
    "#76B7B2",  # cyan
    "#FF9DA7",  # pink
    "#79706E"   # gray
]

# 按 PC 数字自然排序，例如 PC92, PC94, PC103...
def natural_key(x):
    m = re.search(r"\d+", str(x))
    return int(m.group()) if m else str(x)

sample_order = sorted(
    adata.obs[sample_key].dropna().astype(str).unique(),
    key=natural_key
)

if len(sample_order) > len(sample_colors):
    raise ValueError(
        f"sample 数量为 {len(sample_order)}，但 sample_colors 只有 {len(sample_colors)} 个，请增加颜色。"
    )

sample_palette = {
    sample: sample_colors[i]
    for i, sample in enumerate(sample_order)
}

adata.obs[sample_key] = pd.Categorical(
    adata.obs[sample_key].astype(str),
    categories=sample_order,
    ordered=True
)

print("Sample color mapping:")
for k, v in sample_palette.items():
    print(f"{k}: {v}")

# =========================
# 2. 绘图：保持你之前的设置
# =========================
plt.figure(figsize=(10, 9))

name = "sample"

fig = ov.pl.embedding(
    adata,
    basis="X_mde",
    color=[sample_key],
    palette=sample_palette,
    size=100,                    # 保持不变
    alpha=0.7,                   # 保持不变
    frameon="small",
    legend_loc="right margin",
    legend_fontsize=10,
    legend_fontweight="normal",
    add_outline=False,
    show=False,
    return_fig=True
)

# =========================
# 3. 保存
# =========================
save_dir = r"F:\Figures\PlaceCells\FigS7"
os.makedirs(save_dir, exist_ok=True)

fig.savefig(f"{save_dir}/{name}.svg", bbox_inches="tight")
fig.savefig(f"{save_dir}/{name}.png", dpi=600, bbox_inches="tight")

plt.show()
plt.close(fig)

In [ ]:
import os

plt.figure(figsize=(10, 9))

palette = {
    "nonplace": "#E2A528",
    "place": "#6FA3C6",
    "silent": "#7F7F7F"
}
name = "condition"
fig = ov.pl.embedding(
    adata,
    basis="X_mde",
    color=['cell'],
    palette=palette,
    color_map = cmap_custom,
    size=100,
    alpha=0.7,
    frameon="small",
    legend_loc="right margin",
    legend_fontsize=10,
    legend_fontweight="normal",
    add_outline=False,
    show=False,
    return_fig=True,
    vmax=6.0
)

save_dir = r"F:\Figures\PlaceCells\Figure3"
os.makedirs(save_dir, exist_ok=True)

fig.savefig(f"{save_dir}/{name}.svg", bbox_inches="tight")
fig.savefig(f"{save_dir}/{name}.png", dpi=600, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
import os

plt.figure(figsize=(10, 9))

name = "Prox1"
fig = ov.pl.embedding(
    adata,
    basis="X_mde",
    color=name,
    palette=palette,
    color_map = cmap_custom,
    size=100,
    alpha=0.7,
    frameon="small",
    legend_loc="right margin",
    legend_fontsize=10,
    legend_fontweight="normal",
    add_outline=False,
    show=False,
    return_fig=True,
    vmax=6.0
)

save_dir = r"F:\Figures\PlaceCells\FigS7"
os.makedirs(save_dir, exist_ok=True)

fig.savefig(f"{save_dir}/{name}.svg", bbox_inches="tight")
fig.savefig(f"{save_dir}/{name}.png", dpi=600, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
ov.pl.embedding(adata,
                basis='X_mde',
                # color=marker_genes['Neuron'],
                color = ['Grik4','Htr1a'],
                color_map= cmap_custom,
                frameon='small')

In [ ]:
import os

plt.figure(figsize=(10, 9))
palette = {
    "CA1-ProS": "#E2A528",
    "GABAergic-like": "#6FA3C6",
    "Oligodendrocyte": "#7F7F7F"
}

name = "cell_types"
fig = ov.pl.embedding(
    adata,
    basis="X_mde",
    color='cell_types',
    palette=palette,
    # color_map = cmap_custom,
    size=100,
    alpha=0.7,
    frameon="small",
    legend_loc="right margin",
    legend_fontsize=10,
    legend_fontweight="normal",
    add_outline=False,
    show=False,
    return_fig=True,
    vmax=6.0
)

save_dir = r"F:\Figures\PlaceCells\Figure3"
os.makedirs(save_dir, exist_ok=True)

fig.savefig(f"{save_dir}/{name}.svg", bbox_inches="tight")
fig.savefig(f"{save_dir}/{name}.png", dpi=600, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize = (2,4))
ov.pl.cellproportion(adata=adata,celltype_clusters='cell_types',
                    groupby='condition',legend=True,ax=ax)
save_dir = r"F:\Figures\PlaceCells\FigS7"
os.makedirs(save_dir, exist_ok=True)

fig.savefig(f"{save_dir}/Proportion.svg", bbox_inches="tight")
fig.savefig(f"{save_dir}/Proportion.png", dpi=600, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
import pandas as pd

# 1) 查看当前 cell_types
print(adata.obs["cell_types"].value_counts())
print(adata.obs["cell_types"].unique())

# 2) 重命名 cell type
rename_map = {
    "Oligo": "Oligodendrocyte",
    "oligo": "Oligodendrocyte",
    "Unassigned": "GABAergic-like",
    "unassigned": "GABAergic-like",
}

adata.obs["cell_types"] = (
    adata.obs["cell_types"]
    .astype(str)
    .replace(rename_map)
)

# 3) 指定显示顺序
celltype_order = [
    "CA1-ProS",
    "GABAergic-like",
    "Oligodendrocyte",
]

adata.obs["cell_types"] = pd.Categorical(
    adata.obs["cell_types"],
    categories=celltype_order,
    ordered=True
)

# 4) 检查修改结果
print(adata.obs["cell_types"].value_counts())
print(adata.obs["cell_types"].cat.categories)

In [ ]:
marker_genes = {

    "Neuron": ["Rbfox3", "Meg3", "Syt1", "Snap25"],

    "Glutamatergic neuron": ["Slc17a7", "Sv2b", "Neurod2"],

    "GABAergic neuron": ["Gad1", "Gad2", "Slc32a1"],

    "Oligodendrocyte": ["Cldn11", "Plp1", "Mog"],

}

In [ ]:
import os
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from matplotlib.offsetbox import TextArea

# 最小增加：Nature / SVG 设置
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["font.size"] = 7
mpl.rcParams["font.weight"] = "normal"
mpl.rcParams["axes.labelweight"] = "normal"
mpl.rcParams["axes.titleweight"] = "normal"
mpl.rcParams["svg.fonttype"] = "none"   # SVG 文字可编辑
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# 1) 固定 cell type 显示顺序
celltype_order = [
    "CA1-ProS",
    "GABAergic-like",
    "Oligodendrocyte",
]

adata.obs["cell_types"] = pd.Categorical(
    adata.obs["cell_types"].astype(str),
    categories=celltype_order,
    ordered=True
)

# 2) marker gene 顺序：把 Oligodendrocyte 放最后
marker_genes = {
    "Neuron": [
        "Rbfox3", "Syt1", "Snap25"
    ],

    "Glutamatergic neuron": [
        "Slc17a7", "Sv2b", "Neurod2"
    ],

    "GABAergic neuron": [
        "Gad1", "Gad2", "Slc32a1"
    ],

    "Oligodendrocyte": [
        "Cldn11", "Plp1", "Mog"
    ],
}

# 3) 过滤不存在的基因，避免 KeyError
marker_genes_filtered = {
    group: [g for g in genes if g in adata.var_names]
    for group, genes in marker_genes.items()
}

marker_genes_filtered = {
    group: genes
    for group, genes in marker_genes_filtered.items()
    if len(genes) > 0
}

print(marker_genes_filtered)

save_dir = r"F:\Figures\PlaceCells\FigS7"
os.makedirs(save_dir, exist_ok=True)

# 4) 画图：最小修改是不要直接 save，而是 return_fig
fig = ov.pl.dotplot(
    adata,
    marker_genes_filtered,
    groupby="cell_types",
    categories_order=celltype_order,
    standard_scale="var",
    preserve_dict_order=True,
    show=False,
    return_fig=True,
    cmap=cmap_custom,
)

if fig is None:
    fig = plt.gcf()

# 5) 保存前修复 SVG fontweight=100 报错
for text in fig.findobj(match=mtext.Text):
    text.set_fontweight("normal")
    text.set_fontfamily("Arial")

for obj in fig.findobj():
    if isinstance(obj, TextArea):
        obj._text.set_fontweight("normal")
        obj._text.set_fontfamily("Arial")

# 6) 保存 SVG 和 PNG
svg_path = os.path.join(save_dir, "Dotplot.svg")
png_path = os.path.join(save_dir, "Dotplot.png")

fig.savefig(
    svg_path,
    format="svg",
    bbox_inches="tight",
    transparent=False
)

fig.savefig(
    png_path,
    format="png",
    dpi=600,
    bbox_inches="tight",
    transparent=False
)

plt.close(fig)

print("Saved SVG:", svg_path)
print("Saved PNG:", png_path)